# Cell 0 — Git Workflow Guide

## Three-Location Workflow

This notebook is part of a three-location git workflow:
**VS Code (author)** → **GitHub (sync)** → **Colab (compute)** → **GitHub (push back)** → **VS Code (pull)**.

Branch: `qat-wa4-extension`. **Never push to `main` directly.**

---

### Step A — Before opening Colab (run in VS Code terminal)

```bash
git checkout -b qat-wa4-extension
git add training/qat_finetune.py notebooks/qat_colab.ipynb
git commit -m "WA-4 ext: QAT finetune script and Colab notebook"
git push origin qat-wa4-extension
```

### Step B — In Colab

1. Run Cell 2 to clone the repo and check out `qat-wa4-extension`.
2. Run Cells 3–9 in order.
3. Once Cell 8 (export) and Cell 9 (GV1K gate) are both green, run Cell 11 to push artifacts back to GitHub.

### Step C — Back in VS Code

```bash
git pull origin qat-wa4-extension
```

This retrieves `checkpoints/best_model_qat.pth` and `onnx_models/streamsense_multihead.qonnx`.

### Step D — Open PR

Open a pull request from `qat-wa4-extension` into `main` **only after Track E confirms the FINN flow works**. Never merge before Track E sign-off.

---

> **Rule:** Never push directly to `main`. All work lives on `qat-wa4-extension` until Track E confirms.

# Project STREAMSENSE — QAT Fine-tuning and QONNX Export

**Branch:** `qat-wa4-extension`

## What this notebook produces

| Artifact | Path | Description |
|---|---|---|
| `best_model_qat.pth` | `checkpoints/` | QAT fine-tuned checkpoint (Brevitas weights + quantizer scales) |
| `streamsense_multihead.qonnx` | `onnx_models/` | Multi-head QONNX — **Track E FINN flow target** |

The QONNX carries all three output heads:
- `logits`        — `[1, 10]`  float32 — identical classification logits
- `embedding`     — `[1, 128]` float32 — linear projection of the 128-dim GAP vector
- `novelty_score` — `[1, 1]`   float32 — `1 − max(softmax(logits))`, always 2-D

This is the Scope 2 multi-head model in QONNX (Brevitas) format, not the WA-4 QDQ INT8 format. Track E's FINN flow requires QONNX; the WA-4 INT8 QDQ model is not FINN-compatible.

## MPIC v1.0 contract (frozen — do not change)

- Input: `[1, 1, 64, 97]` float32, 16 kHz, 64 mel bins, 97 time frames
- Global norm: mean = −30.785545 dB, std = 22.157099 dB
- Opset 17

In [6]:
!git clone https://github.com/bodasingiksheeraja317-svg/STREAMSENSE.git
%cd STREAMSENSE
!git checkout qat-wa4-extension
!git log --oneline -5

fatal: destination path 'STREAMSENSE' already exists and is not an empty directory.
/content/STREAMSENSE
Already on 'qat-wa4-extension'
Your branch is up to date with 'origin/qat-wa4-extension'.
5389529 (HEAD -> qat-wa4-extension, origin/qat-wa4-extension) fix: correct Colab cwd depth in Cell 2, Cell 8, Cell 9
d4402c1 Update QAT fine-tuning
0686dd6 Add streaming wrapper
40d712a WA-4 ext: QAT finetune script and Colab notebook
6d8a6b9 Fix QAT model device handling


In [7]:
# Cell 3 — Install dependencies
# IMPORTANT: This cell installs packages that may change numpy's ABI.
# After this cell completes, you MUST use Runtime → Restart session,
# then run from Cell 2 onward again (skip re-cloning if repo already present).

# Step 1: pin numpy first so all subsequent packages compile/link against it
!pip install -q "numpy<2.0"

# Step 2: install remaining dependencies
!pip install -q "brevitas>=0.10,<0.11" onnx onnxruntime torchaudio

# Step 3: print versions (run this AFTER restarting the runtime)
import torch, brevitas, onnx, onnxruntime, torchaudio, numpy as np
print(f"torch        : {torch.__version__}")
print(f"torchaudio   : {torchaudio.__version__}")
print(f"brevitas     : {brevitas.__version__}")
print(f"onnx         : {onnx.__version__}")
print(f"onnxruntime  : {onnxruntime.__version__}")
print(f"numpy        : {np.__version__}")

torch        : 2.11.0+cu128
torchaudio   : 2.11.0+cu128
brevitas     : 0.10.3
onnx         : 1.22.0
onnxruntime  : 1.27.0
numpy        : 2.0.2


In [1]:
# Cell 4 — Mount Google Drive and copy checkpoint
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

src_ckpt = Path('/content/drive/MyDrive/STREAMSENSE_checkpoints/best_model.pth')
dst_ckpt = Path('checkpoints/best_model.pth')
dst_ckpt.parent.mkdir(parents=True, exist_ok=True)

if src_ckpt.exists():
    import shutil
    shutil.copy(str(src_ckpt), str(dst_ckpt))
    size_mb = dst_ckpt.stat().st_size / 1e6
    print(f"[OK] Copied best_model.pth  ({size_mb:.2f} MB) -> {dst_ckpt}")
else:
    print(f"[WARN] Checkpoint not found at: {src_ckpt}")
    print(f"       Please upload best_model.pth manually via the Colab Files panel")
    print(f"       and place it at: {dst_ckpt}")
    print(f"       Or run:  !cp /content/drive/MyDrive/<YOUR_PATH>/best_model.pth {dst_ckpt}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] Copied best_model.pth  (1.20 MB) -> checkpoints/best_model.pth


In [2]:
# Cell 5 — Verify GPU
!nvidia-smi

import torch
import brevitas

assert torch.cuda.is_available(), "[FAIL] No GPU detected. Change Runtime -> GPU in Colab."
print(f"[PASS] GPU: {torch.cuda.get_device_name(0)}")
print(f"       torch    : {torch.__version__}")
print(f"       brevitas : {brevitas.__version__}")

Tue Jun 23 12:29:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Cell 6 — Download Speech Commands v2
# torchaudio will download and unzip the dataset on first call.

import torchaudio
from pathlib import Path
from collections import Counter

DATA_ROOT = Path('/content/data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_CLASSES = {
    "yes": 0, "no": 1, "up": 2, "down": 3,
    "left": 4, "right": 5, "on": 6, "off": 7,
    "stop": 8, "go": 9,
}

# Download validation split (will also download the archive if absent)
print("Downloading / loading Speech Commands v2 (validation split)...")
val_ds = torchaudio.datasets.SPEECHCOMMANDS(root=str(DATA_ROOT), download=True, subset='validation')
print(f"Validation split total clips : {len(val_ds)}")

# Count clips per target class
counter = Counter()
for _, _, label, *_ in val_ds:
    if label in TARGET_CLASSES:
        counter[label] += 1

print("\nClass distribution (validation, target classes only):")
total_target = 0
for label in sorted(TARGET_CLASSES):
    n = counter.get(label, 0)
    total_target += n
    print(f"  {label:<6} : {n}")
print(f"  TOTAL  : {total_target}")

Validation split total clips : 9981

Class distribution (validation, target classes only):
  down   : 377
  go     : 372
  left   : 352
  no     : 406
  off    : 373
  on     : 363
  right  : 363
  stop   : 350
  up     : 350
  yes    : 397
  TOTAL  : 3703


In [6]:
import os
os.chdir('/content/STREAMSENSE')
!pwd
!ls

/content/STREAMSENSE
 checkpoints	      evaluation_1d	    stats
 checkpoints_1d       golden_vectors	    training
 class_labels.json    MODEL_CARD.md	   'Untitled document-2.md'
 data		      MPIC_v1.0_spec.pdf   'Untitled document.md'
 deployment_package   notebooks
 evaluation	      quick_predict.ipynb


## Cell 7 — QAT Fine-tuning

This cell runs `training/qat_finetune.py` which:

1. Loads `checkpoints/best_model.pth` backbone weights (strict=True).
2. Replaces all `nn.Conv2d` in the backbone with `brevitas.nn.QuantConv2d` (Int8 per-tensor).
3. Replaces all `nn.Linear` in backbone classifier and `embed_head` with `brevitas.nn.QuantLinear`.
4. Applies the Brevitas device-placement fix (`model.to(device)` last, then buffer verification).
5. **Epochs 1–3**: backbone frozen — only `embed_head` weights and Brevitas quantizer scale factors are trained.
6. **Epoch 4+**: all parameters unfrozen for full QAT fine-tuning.
7. Saves best checkpoint by validation top-1 accuracy.
8. Runs the GV1K gate after training — hard exit if top-1 < 90 %.

Expected runtime on T4: ~20–40 minutes for 10 epochs.

In [7]:
# Cell 7 — Run QAT fine-tuning
!python training/qat_finetune.py \
    --ckpt checkpoints/best_model.pth \
    --data /content/data \
    --epochs 10 \
    --lr 1e-5 \
    --out checkpoints/best_model_qat.pth \
    --gvk golden_vectors_1000/normalized \
    --device cuda

STREAMSENSE — QAT Fine-tuning  (Scope 2 QAT extension)
  Checkpoint  : checkpoints/best_model.pth
  Data root   : /content/data
  Epochs      : 10
  LR          : 1e-05
  Output      : checkpoints/best_model_qat.pth
  Device      : cuda
  GV1K dir    : golden_vectors_1000/normalized

[Step 1] Building QAT model...
[build_qat_model] Loaded backbone from epoch 26  val_acc=96.11%
[build_qat_model] All buffers verified on device='cuda'

[Step 2] Loading Speech Commands datasets...
[SpeechCommandsDataset] subset='validation'  kept 3703 clips  (target classes only)
[SpeechCommandsDataset] subset='testing'  kept 4074 clips  (target classes only)
[SpeechCommandsDataset] subset='validation'  kept 3703 clips  (target classes only)
  Train batches : 57
  Val   batches : 58

[Step 3] Training loop
  Epochs 1–3  : backbone FROZEN, training embed_head + quantizer scales
  Epoch  4+   : all parameters UNFROZEN

  [Epoch 1] Backbone FROZEN.  Trainable params: 16,528
/usr/local/lib/python3.12/dist-pack

In [3]:
# Fix: onnxscript version incompatibility with torch.onnx internal API
# The standalone onnxscript installed in earlier cells conflicts with
# the version torch was compiled against. Uninstall it so torch uses
# its own internal copy.
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "onnxscript"],
    capture_output=True, text=True
)
print(result.stdout or "onnxscript uninstalled (or was not installed standalone)")

# Now verify the import chain works
import importlib
# Force torch.onnx to reimport cleanly after env change
for mod in list(sys.modules.keys()):
    if 'onnxscript' in mod or 'torch.onnx._internal' in mod:
        del sys.modules[mod]

import torch.onnx
print("torch.onnx import: OK")

Found existing installation: onnxscript 0.7.0
Uninstalling onnxscript-0.7.0:
  Successfully uninstalled onnxscript-0.7.0

torch.onnx import: OK


In [5]:
# Step 1: confirm onnxscript situation
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "show", "onnxscript"],
                   capture_output=True, text=True)
print(r.stdout if r.stdout else "onnxscript: not installed standalone — GOOD")

# Step 2: confirm clean import
import torch.onnx
print("torch.onnx: OK")

# Step 3: confirm brevitas export import
from brevitas.export import export_qonnx
print("brevitas.export.export_qonnx: OK")

onnxscript: not installed standalone — GOOD
torch.onnx: OK
brevitas.export.export_qonnx: OK


In [7]:
!pip install -q qonnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 895.9/895.9 kB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.0/344.0 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.5/423.5 kB 41.8 MB/s eta 0:00:00


In [14]:
# Cell 8 — Export QONNX (full inline implementation)
#
# This cell does NOT import from qat_finetune.py.  All Brevitas replacements
# are written out in full here so the export is self-contained and reproducible
# without re-running training.

import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

import onnx
import onnxruntime as ort

import brevitas.nn as qnn
from brevitas.quant import Int8WeightPerTensorFloat, Int8ActPerTensorFloat
from brevitas.export import export_qonnx

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.onnx_exec import execute_onnx
from qonnx.transformation.infer_shapes import InferShapes

# ── Project paths ─────────────────────────────────────────────────────────────
ROOT = Path('/content/STREAMSENSE')
sys.path.insert(0, str(ROOT / 'training'))

from model import StreamSenseNet
from streaming_wrapper import StreamSenseWrapper, NUM_CLASSES, EMBEDDING_DIM

CKPT_PATH   = ROOT / 'checkpoints' / 'best_model_qat.pth'
ORIG_CKPT   = ROOT / 'checkpoints' / 'best_model.pth'
OUT_DIR     = ROOT / 'onnx_models'
EXPORT_PATH = OUT_DIR / 'streamsense_multihead.qonnx'

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Brevitas module replacement functions ─────────────────────────────────────

def _replace_conv2d(module):
    for name, child in module.named_children():
        if isinstance(child, nn.Conv2d):
            qconv = qnn.QuantConv2d(
                in_channels  = child.in_channels,
                out_channels = child.out_channels,
                kernel_size  = child.kernel_size,
                stride       = child.stride,
                padding      = child.padding,
                dilation     = child.dilation,
                groups       = child.groups,
                bias         = child.bias is not None,
                weight_quant = Int8WeightPerTensorFloat,
                input_quant  = Int8ActPerTensorFloat,
                output_quant = Int8ActPerTensorFloat,
                return_quant_tensor = False,
            )
            with torch.no_grad():
                qconv.weight.copy_(child.weight)
                if child.bias is not None and qconv.bias is not None:
                    qconv.bias.copy_(child.bias)
            setattr(module, name, qconv)
        else:
            _replace_conv2d(child)
    return module


def _replace_linear(module):
    for name, child in module.named_children():
        if isinstance(child, nn.Linear):
            qlin = qnn.QuantLinear(
                in_features  = child.in_features,
                out_features = child.out_features,
                bias         = child.bias is not None,
                weight_quant = Int8WeightPerTensorFloat,
                input_quant  = Int8ActPerTensorFloat,
                output_quant = Int8ActPerTensorFloat,
                return_quant_tensor = False,
            )
            with torch.no_grad():
                qlin.weight.copy_(child.weight)
                if child.bias is not None and qlin.bias is not None:
                    qlin.bias.copy_(child.bias)
            setattr(module, name, qlin)
        else:
            _replace_linear(child)
    return module


# ── Build model structure (CPU — export always runs on CPU) ───────────────────
device = torch.device('cpu')
model  = StreamSenseWrapper(num_classes=NUM_CLASSES)

orig_ckpt = torch.load(ORIG_CKPT, map_location='cpu', weights_only=True)
model.backbone.load_state_dict(orig_ckpt['model_state'], strict=True)
print(f"[setup] Loaded backbone from epoch {orig_ckpt.get('epoch','?')}")

_replace_conv2d(model.backbone.block1)
_replace_conv2d(model.backbone.block2)
_replace_conv2d(model.backbone.block3)
_replace_linear(model.backbone.classifier)
_replace_linear(model.embed_head)

model.to(device)

for buf_name, buf in model.named_buffers():
    assert buf.device.type == device.type, (
        f"[device-check] Buffer {buf_name!r} is on {buf.device.type!r}, "
        f"expected {device.type!r}. Brevitas device-placement bug."
    )
print("[device-check] All buffers on CPU — OK")

qat_ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=True)
model.load_state_dict(qat_ckpt['model_state'])
print(f"[setup] Loaded QAT checkpoint  epoch={qat_ckpt.get('epoch','?')}  "
      f"val_acc={qat_ckpt.get('val_accuracy', float('nan')):.2f}%")

model.eval()

# ── Export QONNX ──────────────────────────────────────────────────────────────
input_t = torch.zeros(1, 1, 64, 97, dtype=torch.float32)

print(f"\n[export] Exporting QONNX -> {EXPORT_PATH}")
export_qonnx(
    module      = model,
    input_t     = input_t,
    export_path = str(EXPORT_PATH),
)
print(f"[export] Done.")

# ── ONNX structural check ─────────────────────────────────────────────────────
qonnx_model = onnx.load(str(EXPORT_PATH))
onnx.checker.check_model(qonnx_model)
print("[onnx.checker] PASS — model is structurally valid")

# ── QONNX runtime sanity inference ───────────────────────────────────────────
qonnx_wrap = ModelWrapper(str(EXPORT_PATH))
qonnx_wrap = qonnx_wrap.transform(InferShapes())

dummy_np   = np.zeros((1, 1, 64, 97), dtype=np.float32)
input_name = qonnx_wrap.graph.input[0].name
odict      = execute_onnx(qonnx_wrap, {input_name: dummy_np})

output_names = [o.name for o in qonnx_wrap.graph.output]
print(f"[sanity] Output names: {output_names}")
for name, val in odict.items():
    print(f"  {name}: shape={val.shape}")

logits        = odict[output_names[0]]
embedding     = odict[output_names[1]]
novelty_score = odict[output_names[2]]

# ── Shape assertions ──────────────────────────────────────────────────────────
logits_ok  = logits.shape        == (1, 10)
embed_ok   = embedding.shape     == (1, 128)
novelty_ok = novelty_score.shape == (1, 1)

print(f"\n[shape assert] logits        {tuple(logits.shape)}    : {'PASS' if logits_ok  else 'FAIL — expected (1, 10)'}")
print(f"[shape assert] embedding     {tuple(embedding.shape)} : {'PASS' if embed_ok   else 'FAIL — expected (1, 128)'}")
print(f"[shape assert] novelty_score {tuple(novelty_score.shape)}    : {'PASS' if novelty_ok else 'FAIL — expected (1, 1), check keepdim=True'}")

assert logits_ok,  f"logits shape {logits.shape} != (1,10)  — ERR v1.0 contract broken"
assert embed_ok,   f"embedding shape {embedding.shape} != (1,128) — ERR v1.0 contract broken"
assert novelty_ok, f"novelty_score shape {novelty_score.shape} != (1,1) — ERR v1.0 contract broken"

print("\n[PASS] All three output shapes conform to ERR v1.0 contract.")

# ── File size ─────────────────────────────────────────────────────────────────
size_mb = EXPORT_PATH.stat().st_size / 1e6
print(f"[export] File size: {size_mb:.2f} MB  -> {EXPORT_PATH}")

[setup] Loaded backbone from epoch 26
[device-check] All buffers on CPU — OK
[setup] Loaded QAT checkpoint  epoch=9  val_acc=97.11%

[export] Exporting QONNX -> /content/STREAMSENSE/onnx_models/streamsense_multihead.qonnx
[export] Done.
[onnx.checker] PASS — model is structurally valid
[sanity] Output names: ['logits', '143', '147']
  logits: shape=(1, 10)
  143: shape=(1, 128)
  147: shape=(1, 1)

[shape assert] logits        (1, 10)    : PASS
[shape assert] embedding     (1, 128) : PASS
[shape assert] novelty_score (1, 1)    : PASS

[PASS] All three output shapes conform to ERR v1.0 contract.
[export] File size: 1.28 MB  -> /content/STREAMSENSE/onnx_models/streamsense_multihead.qonnx


In [16]:
# Cell 8a — Generate GV1K vectors from Speech Commands test split
#
# The golden_vectors_1000/normalized/ directory exists in the repo but
# contains NO .bin files — they are ~25 MB of derived binary artifacts
# that were never committed to Git.
#
# This cell generates them now using the MPIC v1.0 pipeline, drawing
# 100 samples per class (1000 total) from the Speech Commands test split.
# Reproducible: fixed random seed 42 (same as generate_golden_1000.py).

import random
import numpy as np
import torch
import torchaudio
import torchaudio.transforms as T
from pathlib import Path

# ── MPIC v1.0 frozen constants (must match qat_finetune.py exactly) ──────────
SAMPLE_RATE   = 16000
FRAME_LEN     = 16000
N_FFT         = 512
HOP_LENGTH    = 160
N_MELS        = 64
CENTER        = False
POWER         = 2.0
LOG_EPS       = 1e-10
CLIP_FLOOR_DB = -80.0
GLOBAL_MEAN   = -30.785545
GLOBAL_STD    = 22.157099
EXPECTED_T    = (FRAME_LEN - N_FFT) // HOP_LENGTH + 1   # 97

TARGET_CLASSES = {
    "yes": 0, "no": 1, "up": 2, "down": 3,
    "left": 4, "right": 5, "on": 6, "off": 7,
    "stop": 8, "go": 9,
}

N_PER_CLASS = 100
RANDOM_SEED = 42

ROOT     = Path('/content/STREAMSENSE')
GV1K_DIR = ROOT / 'golden_vectors_1000' / 'normalized'
GV1K_DIR.mkdir(parents=True, exist_ok=True)

# Skip if already generated
existing = list(GV1K_DIR.glob('*_norm.bin'))
if len(existing) >= 1000:
    print(f"[OK] GV1K vectors already present: {len(existing)} files — skipping generation.")
else:
    print(f"[INFO] Generating GV1K vectors (this may take ~2-3 minutes)...")

    # ── Build mel transform ───────────────────────────────────────────────────
    mel_transform = T.MelSpectrogram(
        sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, window_fn=torch.hann_window, center=CENTER, power=POWER,
    )

    def preprocess_to_norm(waveform, sr):
        """Waveform tensor [C, T] -> normalised [64, 97] numpy float32."""
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        L = waveform.shape[1]
        if L < FRAME_LEN:
            waveform = torch.nn.functional.pad(waveform, (0, FRAME_LEN - L))
        elif L > FRAME_LEN:
            waveform = waveform[:, :FRAME_LEN]
        mel = mel_transform(waveform)                   # [1, 64, 97]
        mel = 10.0 * torch.log10(mel + LOG_EPS)
        mel = torch.clamp(mel, min=CLIP_FLOOR_DB)
        norm = (mel - GLOBAL_MEAN) / GLOBAL_STD
        return norm.squeeze(0).numpy().astype(np.float32)   # [64, 97]

    # ── Load test split, bucket by class ─────────────────────────────────────
    print("Loading Speech Commands test split...")
    test_ds = torchaudio.datasets.SPEECHCOMMANDS(
        root='/content/data', download=True, subset='testing'
    )

    buckets = {label: [] for label in TARGET_CLASSES}
    for item in test_ds:
        waveform, sr, label = item[0], item[1], item[2]
        if label in TARGET_CLASSES:
            buckets[label].append((waveform, sr))

    for label, items in buckets.items():
        print(f"  {label:<6}: {len(items)} clips available")

    # ── Sample N_PER_CLASS per class ──────────────────────────────────────────
    rng = random.Random(RANDOM_SEED)
    gv_idx = 0
    n_written = 0

    for label, class_idx in TARGET_CLASSES.items():
        items = buckets[label][:]
        rng.shuffle(items)
        selected = items[:N_PER_CLASS]

        for waveform, sr in selected:
            norm = preprocess_to_norm(waveform, sr)
            assert norm.shape == (N_MELS, EXPECTED_T), f"Shape error: {norm.shape}"
            assert norm.dtype == np.float32

            fname = GV1K_DIR / f"GV1K_{gv_idx:04d}_{label}_norm.bin"
            norm.tofile(str(fname))
            gv_idx += 1
            n_written += 1

    n_final = len(list(GV1K_DIR.glob('*_norm.bin')))
    print(f"\n[{'OK' if n_final == 1000 else 'WARN'}] Written: {n_final} *_norm.bin files to {GV1K_DIR}")
    if n_final < 900:
        raise RuntimeError(f"Only {n_final} GV1K vectors generated — too few to pass the 90% gate. Check data.")
    print("[INFO] GV1K generation complete. Ready for Cell 9.")

[INFO] Generating GV1K vectors (this may take ~2-3 minutes)...
Loading Speech Commands test split...
  yes   : 419 clips available
  no    : 405 clips available
  up    : 425 clips available
  down  : 406 clips available
  left  : 412 clips available
  right : 396 clips available
  on    : 396 clips available
  off   : 402 clips available
  stop  : 411 clips available
  go    : 402 clips available

[OK] Written: 1000 *_norm.bin files to /content/STREAMSENSE/golden_vectors_1000/normalized
[INFO] GV1K generation complete. Ready for Cell 9.


In [17]:
# Cell 9 — GV1K top-1 verification on exported QONNX

import sys
import numpy as np
from pathlib import Path

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.onnx_exec import execute_onnx
from qonnx.transformation.infer_shapes import InferShapes

ROOT       = Path('/content/STREAMSENSE')
QONNX_PATH = ROOT / 'onnx_models' / 'streamsense_multihead.qonnx'
GV1K_DIR   = ROOT / 'golden_vectors_1000' / 'normalized'

TARGET_CLASSES = {
    "yes": 0, "no": 1, "up": 2, "down": 3,
    "left": 4, "right": 5, "on": 6, "off": 7,
    "stop": 8, "go": 9,
}


def _parse_label(stem: str):
    parts = stem.split('_')
    if len(parts) < 4:
        return None
    return TARGET_CLASSES.get(parts[2].lower(), None)


# ── Load QONNX model (once, before the loop) ─────────────────────────────────
qonnx_wrap   = ModelWrapper(str(QONNX_PATH))
qonnx_wrap   = qonnx_wrap.transform(InferShapes())
input_name   = qonnx_wrap.graph.input[0].name
output_names = [o.name for o in qonnx_wrap.graph.output]

print(f"QONNX input : {input_name!r}")
print(f"Outputs     : {output_names}")

# ── Load GV1K vectors ─────────────────────────────────────────────────────────
bin_files = sorted(GV1K_DIR.glob('*_norm.bin'))
print(f"\nGV1K vectors found : {len(bin_files)}")

if len(bin_files) == 0:
    print(f"[WARN] No *_norm.bin files found in {GV1K_DIR}")
    print("       Generate GV1K vectors first: python training/generate_golden_1000.py")

# ── Run inference on all 1000 vectors ────────────────────────────────────────
correct = 0
wrong   = 0
skipped = 0

for bf in bin_files:
    true_idx = _parse_label(bf.stem)
    if true_idx is None:
        skipped += 1
        continue

    raw = np.fromfile(str(bf), dtype='<f4')
    if raw.size != 64 * 97:
        skipped += 1
        continue

    inp   = raw.reshape(1, 1, 64, 97).astype(np.float32)
    odict = execute_onnx(qonnx_wrap, {input_name: inp})

    logits = odict[output_names[0]]          # [1, 10]
    pred   = int(np.argmax(logits[0]))

    if pred == true_idx:
        correct += 1
    else:
        wrong += 1

total_checked = correct + wrong
top1_acc      = 100.0 * correct / total_checked if total_checked > 0 else 0.0

print(f"\n{'='*50}")
print(f"GV1K QONNX Verification")
print(f"{'='*50}")
print(f"  Total checked  : {total_checked}")
print(f"  Correct        : {correct}")
print(f"  Wrong          : {wrong}")
print(f"  Skipped        : {skipped}")
print(f"  Top-1 accuracy : {top1_acc:.2f}%")

gate_passed = (top1_acc >= 90.0)
print(f"\n[{'PASS' if gate_passed else 'FAIL'}] GV1K gate {'passed' if gate_passed else 'FAILED'} "
      f"({top1_acc:.2f}% {'≥' if gate_passed else '<'} 90.0%)")

assert gate_passed, (
    f"GV1K gate FAILED: {top1_acc:.2f}% < 90.0% minimum.  "
    f"Do NOT push — debug the QONNX export first."
)

QONNX input : 'x.861'
Outputs     : ['logits', '143', '147']

GV1K vectors found : 1000

GV1K QONNX Verification
  Total checked  : 1000
  Correct        : 975
  Wrong          : 25
  Skipped        : 0
  Top-1 accuracy : 97.50%

[PASS] GV1K gate passed (97.50% ≥ 90.0%)


## Cell 10 — Review before pushing

**Before running Cell 11**, confirm the following in the Cell 8 and Cell 9 outputs:

- `[PASS] All three output shapes conform to ERR v1.0 contract.`
  - `logits` : `(1, 10)`
  - `embedding` : `(1, 128)`
  - `novelty_score` : `(1, 1)` — must be exactly 2-D
- `[PASS] GV1K gate passed (≥ 90.0%)`
- `[onnx.checker] PASS`

If **any** of these is red or shows FAIL, **do not run Cell 11**. Debug the export first:
- If `novelty_score` is `(1,)` (1-D), check that `keepdim=True` is present in `StreamSenseWrapper.forward()`.
- If GV1K accuracy is < 90 %, re-run training with more epochs or a slightly higher learning rate.
- If the ONNX checker fails, check that the Brevitas version is compatible with the installed PyTorch.

In [18]:
# Cell 11 — Git commit and push artifacts back to GitHub
#
# REPLACE the email, name, and token/username before running.
# If push fails with authentication error, set the remote URL with your PAT:
#   !git remote set-url origin https://<TOKEN>@github.com/<USERNAME>/STREAMSENSE.git

!git config user.email "your.email@example.com"   # REPLACE with your email
!git config user.name  "Your Name"                 # REPLACE with your name

!git add checkpoints/best_model_qat.pth
!git add onnx_models/streamsense_multihead.qonnx
!git add notebooks/qat_colab.ipynb

!git commit -m "WA-4 ext: QAT checkpoint and QONNX multihead export — 3-output Track E target — GV1K green"
!git push origin qat-wa4-extension

print()
print("If the push failed due to authentication, run this in a code cell:")
print("  !git remote set-url origin https://<TOKEN>@github.com/<USERNAME>/STREAMSENSE.git")
print("Then re-run this cell.")

The following paths are ignored by one of your .gitignore files:
onnx_models
hint: Use -f if you really want to add them.
hint: Turn this message off by running
hint: "git config advice.addIgnoredFile false"
[qat-wa4-extension 8d7a608] WA-4 ext: QAT checkpoint and QONNX multihead export — 3-output Track E target — GV1K green
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 checkpoints/best_model_qat.pth
fatal: could not read Username for 'https://github.com': No such device or address

If the push failed due to authentication, run this in a code cell:
  !git remote set-url origin https://<TOKEN>@github.com/<USERNAME>/STREAMSENSE.git
Then re-run this cell.


In [19]:
# Cell 12 — Copy artifacts to Google Drive for local backup
import shutil
from pathlib import Path

drive_out = Path('/content/drive/MyDrive/STREAMSENSE_outputs')
drive_out.mkdir(parents=True, exist_ok=True)

src_ckpt  = Path('checkpoints/best_model_qat.pth')
src_qonnx = Path('onnx_models/streamsense_multihead.qonnx')
dst_ckpt  = drive_out / 'best_model_qat.pth'
dst_qonnx = drive_out / 'streamsense_multihead.qonnx'

shutil.copy(str(src_ckpt),  str(dst_ckpt))
shutil.copy(str(src_qonnx), str(dst_qonnx))

print(f"[copy] best_model_qat.pth          : {dst_ckpt.stat().st_size / 1e6:.2f} MB  -> {dst_ckpt}")
print(f"[copy] streamsense_multihead.qonnx : {dst_qonnx.stat().st_size / 1e6:.2f} MB  -> {dst_qonnx}")

[copy] best_model_qat.pth          : 1.34 MB  -> /content/drive/MyDrive/STREAMSENSE_outputs/best_model_qat.pth
[copy] streamsense_multihead.qonnx : 1.28 MB  -> /content/drive/MyDrive/STREAMSENSE_outputs/streamsense_multihead.qonnx


## Cell 14 — Handover Summary

| File | Path | For whom | Purpose |
|---|---|---|---|
| `streamsense_multihead.qonnx` | `onnx_models/` | Track E | FINN flow target — all 3 heads, QONNX format (Brevitas QAT) |
| `streamsense_multihead_fp32.onnx` | `onnx_models/` | Track E | FP32 reference — ORT inference, all 3 heads |
| `best_model_qat.pth` | `checkpoints/` | Internal Track A | QAT checkpoint — source of QONNX, Brevitas quantizer scales included |
| `streamsense_multihead_int8.onnx` | `onnx_models/` | Internal Track A | QDQ PTQ reference — 95.8% GV1K, NOT FINN-compatible |